# Data Visualization for Data Analysis Bootcamp

A hands-on Jupyter notebook covering **30 practical visualization concepts** with Matplotlib and Seaborn. Each topic has exactly two cells: a Markdown explanation followed by a code cell with `#` comments.

**Requirements:** `numpy`, `pandas`, `matplotlib`, and `seaborn`. If needed, install them in a notebook cell with `%pip install numpy pandas matplotlib seaborn`.

All examples use a synthetic sales dataset created in the first lesson; no external files are required. Run the cells from top to bottom. The notebook ends with a four-panel sales dashboard.

## Learning path
1. Chart choice and Matplotlib basics
2. Category comparisons and distributions
3. Relationships, correlations, and heatmaps
4. Uncertainty, annotations, styling, and saving
5. A mini dashboard project

## 1. Visualization Setup and Sample Dataset

Data visualization turns data into a form people can interpret quickly. **Matplotlib** provides detailed control over figures, axes, labels, and annotations; **Seaborn** builds on Matplotlib to make statistical charts easier.

This notebook creates a small sales dataset directly in Python. Later lessons reuse it to compare revenue, products, regions, customers, and dates. No external file is needed.

In [ ]:
import numpy as np  # Import NumPy for numerical calculations and reproducible sample data.
import pandas as pd  # Import pandas to organize records in a table.
import matplotlib.pyplot as plt  # Import Matplotlib's plotting interface.
import seaborn as sns  # Import Seaborn for statistical visualizations.

rng = np.random.default_rng(42)  # Create a random generator so sample values are reproducible.
dates = pd.date_range("2025-01-01", periods=120, freq="D")  # Create 120 consecutive daily dates.
products = np.array(["Laptop", "Phone", "Tablet", "Monitor"])  # Define the product categories.
regions = np.array(["North", "South", "East", "West"])  # Define the sales regions.
channels = np.array(["Online", "Store", "Partner"])  # Define the sales channels.
df = pd.DataFrame({  # Build one row per sample transaction.
    "date": rng.choice(dates, size=400),  # Assign each transaction a date from the sample period.
    "product": rng.choice(products, size=400),  # Assign a product to each transaction.
    "region": rng.choice(regions, size=400),  # Assign a region to each transaction.
    "channel": rng.choice(channels, size=400),  # Assign a sales channel.
    "units": rng.integers(1, 8, size=400),  # Generate a number of units from 1 through 7.
    "unit_price": rng.choice([80, 120, 250, 800, 900], size=400),  # Choose a sample unit price.
    "satisfaction": np.clip(rng.normal(4.0, 0.7, size=400), 1, 5).round(1),  # Simulate a satisfaction score from 1 to 5.
})  # Finish creating the table.
df["revenue"] = df["units"] * df["unit_price"]  # Calculate revenue for each transaction.
df["profit"] = df["revenue"] * rng.uniform(0.08, 0.32, size=len(df))  # Simulate an illustrative profit estimate per transaction.
df = df.sort_values("date").reset_index(drop=True)  # Sort records by date and recreate the row index.
print(df.head())  # Inspect the first five records.
print("Shape:", df.shape)  # Display the number of records and columns.

## 2. Choosing the Right Chart

Pick a chart based on the question, not on decoration.

- **Trend over time:** line chart.
- **Compare categories:** bar chart.
- **Distribution:** histogram, box plot, or violin plot.
- **Relationship between numeric variables:** scatter plot.
- **Relationship across many numeric variables:** correlation heatmap or pair plot.
- **Part-to-whole:** stacked bar for comparisons; pie charts are best reserved for a few simple categories.

Good charts use a clear title, readable axes, meaningful units, and as little visual clutter as possible.

In [ ]:
chart_purpose = {  # Map common analysis questions to useful chart types.
    "How does revenue change over time?": "Line chart",  # Lines show ordered time trends.
    "Which product sells the most?": "Bar chart",  # Bars make category comparisons easy.
    "How are order values distributed?": "Histogram or box plot",  # These show distribution and spread.
    "Are units and revenue related?": "Scatter plot",  # A scatter plot shows paired numeric observations.
    "How do numeric metrics move together?": "Correlation heatmap",  # A heatmap summarizes pairwise correlations.
}  # Finish the chart-selection guide.
for question, recommendation in chart_purpose.items():  # Visit each question and its recommended chart.
    print(question, "→", recommendation)  # Display a practical chart recommendation.

## 3. Line Chart: Trend Over Time

A line chart connects observations in order and is useful for trends such as daily revenue, monthly sales, or website traffic.

First aggregate transactions by date. Always sort the time axis chronologically; otherwise, the line may connect dates in the wrong order. A line chart is less appropriate for unrelated categories without a natural order.

In [ ]:
daily_revenue = df.groupby("date", as_index=False)["revenue"].sum().sort_values("date")  # Sum revenue per date and sort chronologically.
fig, ax = plt.subplots(figsize=(10, 4))  # Create a figure and a single plotting axis.
ax.plot(daily_revenue["date"], daily_revenue["revenue"], linewidth=1.8)  # Draw the daily revenue line.
ax.set_title("Daily Revenue Over Time")  # Add a descriptive title.
ax.set_xlabel("Date")  # Label the horizontal axis.
ax.set_ylabel("Revenue (sample currency)")  # Label the vertical axis with its units.
ax.tick_params(axis="x", rotation=45)  # Rotate date labels to reduce overlap.
ax.grid(True, alpha=0.25)  # Add a light grid for easier value comparisons.
fig.tight_layout()  # Adjust spacing so labels fit.
plt.show()  # Render the chart.

## 4. Bar Chart: Compare Categories

A bar chart compares values across discrete categories. Bars should usually start at zero because their length encodes magnitude.

Aggregate the values first—for example, sum revenue by product. Sort the categories when ranking matters, and label axes with the metric and units.

In [ ]:
product_revenue = df.groupby("product", as_index=False)["revenue"].sum().sort_values("revenue", ascending=False)  # Sum revenue and rank products.
fig, ax = plt.subplots(figsize=(8, 4))  # Create the chart canvas and axis.
ax.bar(product_revenue["product"], product_revenue["revenue"])  # Draw one bar for each product.
ax.set_title("Total Revenue by Product")  # Describe the comparison.
ax.set_xlabel("Product")  # Label the category axis.
ax.set_ylabel("Revenue (sample currency)")  # Label the numeric axis.
ax.grid(axis="y", alpha=0.25)  # Add horizontal guide lines.
fig.tight_layout()  # Prevent labels from being cut off.
plt.show()  # Display the bar chart.

## 5. Horizontal Bar Chart

Horizontal bars are a good choice when category names are long or when ranking many categories. They also make labels easier to read.

Use `barh()` for horizontal bars. Because the first bar is drawn at the bottom-to-top order, reverse a ranked table when you want the largest category at the top.

In [ ]:
region_revenue = df.groupby("region")["revenue"].sum().sort_values()  # Sum revenue by region and sort from smallest to largest.
fig, ax = plt.subplots(figsize=(7, 4))  # Create a landscape figure.
ax.barh(region_revenue.index, region_revenue.values)  # Draw a horizontal bar for each region.
ax.set_title("Revenue by Region")  # Add the title.
ax.set_xlabel("Revenue (sample currency)")  # Label the numeric axis.
ax.set_ylabel("Region")  # Label the category axis.
ax.grid(axis="x", alpha=0.25)  # Add vertical guide lines for reading values.
fig.tight_layout()  # Fit labels within the figure.
plt.show()  # Display the horizontal bar chart.

## 6. Grouped Bar Chart

A grouped bar chart compares multiple series within each category—for example, revenue by product, split by sales channel.

Pivot the data into categories and series first, then plot the table. Grouped bars work well for a few series; with too many series, the chart becomes crowded.

In [ ]:
channel_product = df.pivot_table(index="product", columns="channel", values="revenue", aggfunc="sum", fill_value=0)  # Summarize revenue by product and channel.
fig, ax = plt.subplots(figsize=(9, 5))  # Create a figure large enough for the grouped bars.
channel_product.plot(kind="bar", ax=ax, width=0.82)  # Draw a bar group per product with one bar per channel.
ax.set_title("Revenue by Product and Channel")  # Add a descriptive title.
ax.set_xlabel("Product")  # Label the categories.
ax.set_ylabel("Revenue (sample currency)")  # Label the revenue axis.
ax.tick_params(axis="x", rotation=0)  # Keep product labels horizontal.
ax.legend(title="Channel")  # Identify the channel represented by each color.
fig.tight_layout()  # Adjust subplot spacing.
plt.show()  # Display the grouped bar chart.

## 7. Stacked Bar Chart

A stacked bar chart shows both totals and how each total is divided among subcategories. It can reveal each region's channel mix.

Use stacks when the total and composition are both important. Comparing small segments across multiple bars is harder than comparing grouped bars, so consider the question before choosing a stacked design.

In [ ]:
region_channel = df.pivot_table(index="region", columns="channel", values="revenue", aggfunc="sum", fill_value=0)  # Summarize revenue by region and channel.
fig, ax = plt.subplots(figsize=(8, 4))  # Create the figure and plotting axis.
region_channel.plot(kind="bar", stacked=True, ax=ax)  # Stack channel revenue within each regional bar.
ax.set_title("Revenue Composition by Region")  # Add the title.
ax.set_xlabel("Region")  # Label the categories.
ax.set_ylabel("Revenue (sample currency)")  # Label total revenue.
ax.legend(title="Channel")  # Explain the stack categories.
ax.tick_params(axis="x", rotation=0)  # Keep region labels horizontal.
fig.tight_layout()  # Fit all labels and legend.
plt.show()  # Render the stacked chart.

## 8. Histogram: Distribution

A histogram groups numeric values into bins and displays how many observations fall in each bin. It helps reveal skew, clusters, and the spread of a variable.

The number of bins affects the story the chart appears to tell. Use an appropriate range and consistent bins when comparing datasets.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))  # Create a figure and axis for the distribution.
ax.hist(df["revenue"], bins=20, edgecolor="black", alpha=0.8)  # Count revenue observations in 20 bins.
ax.set_title("Distribution of Transaction Revenue")  # Add a title explaining the distribution.
ax.set_xlabel("Revenue (sample currency)")  # Label the horizontal axis.
ax.set_ylabel("Number of transactions")  # Label the frequency axis.
ax.grid(axis="y", alpha=0.2)  # Add subtle horizontal reference lines.
fig.tight_layout()  # Adjust spacing.
plt.show()  # Display the histogram.

## 9. Histogram with Seaborn

Seaborn's `histplot()` offers convenient distribution plots. Setting `kde=True` overlays a smoothed density estimate, which can help show the shape of a distribution.

A KDE curve is an estimate, not the observed data itself. Use it as a supplement to the histogram rather than a replacement when the data is sparse or highly discrete.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))  # Create a figure with one axis.
sns.histplot(data=df, x="satisfaction", bins=10, kde=True, color=plt.rcParams["axes.prop_cycle"].by_key()["color"][0], ax=ax)  # Plot satisfaction scores with a histogram and density estimate.
ax.set_title("Customer Satisfaction Distribution")  # Add a descriptive title.
ax.set_xlabel("Satisfaction score (1–5)")  # Explain the score range.
ax.set_ylabel("Count")  # Label the count axis.
fig.tight_layout()  # Fit labels cleanly.
plt.show()  # Render the Seaborn histogram.

## 10. Box Plot: Spread and Outliers

A box plot summarizes a distribution with the median, quartiles, and whiskers. Points beyond the whisker rule are displayed as potential outliers; they are not automatically errors.

Box plots are useful for comparing distributions across groups. The box represents the middle 50% of the data, while the median line marks its center.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))  # Create the figure and axis.
sns.boxplot(data=df, x="product", y="revenue", ax=ax)  # Compare revenue distributions for each product.
ax.set_title("Revenue Distribution by Product")  # Add the chart title.
ax.set_xlabel("Product")  # Label the categories.
ax.set_ylabel("Revenue (sample currency)")  # Label the numeric measure.
fig.tight_layout()  # Adjust spacing around labels.
plt.show()  # Display the box plot.

## 11. Violin Plot: Distribution Shape

A violin plot combines a density shape with summary information. It can reveal whether a distribution is concentrated around one value or has multiple dense regions.

Violin plots require enough observations to estimate a meaningful density. For small samples, a box plot or strip plot may communicate the data more honestly.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))  # Create the figure and axis.
sns.violinplot(data=df, x="product", y="satisfaction", inner="quartile", cut=0, ax=ax)  # Compare score distributions and show quartiles.
ax.set_title("Satisfaction Distribution by Product")  # Add the title.
ax.set_xlabel("Product")  # Label the category axis.
ax.set_ylabel("Satisfaction score")  # Label the score axis.
fig.tight_layout()  # Adjust the layout.
plt.show()  # Render the violin plot.

## 12. Scatter Plot: Relationship Between Two Variables

A scatter plot displays paired observations for two numeric variables. It can reveal positive or negative association, clusters, non-linear patterns, and unusual points.

A relationship in a scatter plot does not prove that one variable causes the other. Use transparency (`alpha`) when many observations overlap.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))  # Create the figure and axis.
ax.scatter(df["units"], df["revenue"], alpha=0.45)  # Plot each transaction as a units/revenue point.
ax.set_title("Units Sold vs. Revenue")  # Describe the relationship being examined.
ax.set_xlabel("Units sold")  # Label the horizontal variable.
ax.set_ylabel("Revenue (sample currency)")  # Label the vertical variable.
ax.grid(True, alpha=0.2)  # Add subtle reference lines.
fig.tight_layout()  # Prevent labels from being clipped.
plt.show()  # Display the scatter plot.

## 13. Scatter Plot with Category Color

Color can encode a third variable in a scatter plot. Seaborn's `hue` parameter assigns colors according to a category, which helps compare groups within the same numeric relationship.

Use a small number of clearly differentiated groups. Too many colors can make the chart hard to read.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))  # Create the figure and axis.
sns.scatterplot(data=df, x="units", y="revenue", hue="product", alpha=0.7, ax=ax)  # Color each point by product.
ax.set_title("Units vs. Revenue by Product")  # Add the title.
ax.set_xlabel("Units sold")  # Label the x-axis.
ax.set_ylabel("Revenue (sample currency)")  # Label the y-axis.
ax.legend(title="Product")  # Add a legend explaining the product colors.
fig.tight_layout()  # Adjust spacing for the legend and labels.
plt.show()  # Display the grouped scatter plot.

## 14. Bubble Chart: Encode a Third Numeric Variable

A bubble chart uses point size to encode a third numeric variable. For example, x can represent units, y can represent revenue, and bubble size can represent satisfaction or profit.

Size encoding is approximate: large bubbles can dominate the view. Scale bubble sizes deliberately and include a clear explanation of what size means.

In [ ]:
bubble_sizes = 20 + df["profit"] / df["profit"].max() * 280  # Convert profit values into display sizes from about 20 to 300.
fig, ax = plt.subplots(figsize=(8, 5))  # Create the chart canvas.
points = ax.scatter(df["units"], df["revenue"], s=bubble_sizes, alpha=0.35)  # Plot points and encode profit with bubble size.
ax.set_title("Sales: Units vs. Revenue (Bubble Size = Profit)")  # Explain the size encoding in the title.
ax.set_xlabel("Units sold")  # Label the horizontal axis.
ax.set_ylabel("Revenue (sample currency)")  # Label the vertical axis.
ax.grid(True, alpha=0.2)  # Add a subtle grid.
fig.tight_layout()  # Adjust layout.
plt.show()  # Display the bubble chart.

## 15. Count Plot: Frequency of Categories

A count plot shows how many rows belong to each category. It is useful for checking category balance, such as transaction counts by product, region, or channel.

A count plot counts records, not necessarily unique customers or orders. If repeated rows exist, choose the right unit of analysis before interpreting the bars.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))  # Create a figure and axis.
sns.countplot(data=df, x="region", order=df["region"].value_counts().index, color=plt.rcParams["axes.prop_cycle"].by_key()["color"][0], ax=ax)  # Count records per region in descending frequency order.
ax.set_title("Number of Transactions by Region")  # Add a clear title.
ax.set_xlabel("Region")  # Label the region axis.
ax.set_ylabel("Transaction count")  # Label the frequency axis.
fig.tight_layout()  # Adjust spacing.
plt.show()  # Render the count plot.

## 16. Bar Chart with Confidence Intervals

A Seaborn bar plot can summarize a numeric measure by category and show an uncertainty interval around the estimated mean. By default, modern Seaborn versions use a bootstrap confidence interval unless configured otherwise.

Intervals communicate uncertainty, not the range of individual observations. For a total such as revenue, calculate and plot the sum explicitly rather than relying on a bar plot's default mean estimator.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))  # Create the figure and axis.
sns.barplot(data=df, x="product", y="satisfaction", errorbar=("ci", 95), color=plt.rcParams["axes.prop_cycle"].by_key()["color"][0], ax=ax)  # Plot mean satisfaction with a 95% bootstrap confidence interval.
ax.set_title("Average Satisfaction by Product (95% CI)")  # State the statistic and interval in the title.
ax.set_xlabel("Product")  # Label the categories.
ax.set_ylabel("Mean satisfaction score")  # Label the estimate.
fig.tight_layout()  # Adjust spacing.
plt.show()  # Display the bar plot with intervals.

## 17. Heatmap: Cross-Tabulated Values

A heatmap uses color intensity to show values in a matrix. It is especially useful for comparing a metric across two categorical dimensions, such as region by product.

Use annotations when values are few enough to remain readable. Choose a meaningful color scale and label the measure. Missing combinations need careful treatment; zero and missing do not always mean the same thing.

In [ ]:
region_product = df.pivot_table(index="region", columns="product", values="revenue", aggfunc="sum", fill_value=0)  # Sum revenue for every region-product combination.
fig, ax = plt.subplots(figsize=(8, 4))  # Create the heatmap figure and axis.
sns.heatmap(region_product, annot=True, fmt=".0f", cmap="YlGnBu", linewidths=0.5, ax=ax)  # Encode revenue by color and annotate each cell.
ax.set_title("Revenue Heatmap: Region × Product")  # Describe the matrix.
ax.set_xlabel("Product")  # Label the columns.
ax.set_ylabel("Region")  # Label the rows.
fig.tight_layout()  # Fit the chart and labels.
plt.show()  # Render the heatmap.

## 18. Correlation Heatmap

A correlation heatmap summarizes pairwise correlations between numeric variables. Values near 1 indicate strong positive linear association, values near -1 indicate strong negative association, and values near 0 indicate weak linear association.

Correlation is not causation. Also check for outliers, non-linear relationships, and duplicated or derived features before drawing conclusions.

In [ ]:
numeric_df = df[["units", "unit_price", "revenue", "profit", "satisfaction"]]  # Select numeric fields for correlation analysis.
correlations = numeric_df.corr()  # Calculate pairwise Pearson correlation coefficients.
fig, ax = plt.subplots(figsize=(7, 5))  # Create the figure and axis.
sns.heatmap(correlations, annot=True, fmt=".2f", vmin=-1, vmax=1, center=0, cmap="coolwarm", ax=ax)  # Plot coefficients using a fixed -1-to-1 scale.
ax.set_title("Correlation Between Numeric Variables")  # Add a descriptive title.
fig.tight_layout()  # Prevent the labels from clipping.
plt.show()  # Display the correlation heatmap.

## 19. Pair Plot: Many Relationships at Once

A pair plot creates scatter plots for pairs of numeric variables and distribution plots along the diagonal. It is useful for early exploratory data analysis when you want to inspect several variables together.

Pair plots can become slow and cluttered with many columns or rows. Choose a small set of relevant variables and sample large datasets if necessary.

In [ ]:
pair_data = df[["units", "unit_price", "revenue", "satisfaction", "product"]].sample(n=180, random_state=42)  # Select a subset and sample it for a readable pair plot.
pair_grid = sns.pairplot(pair_data, vars=["units", "unit_price", "revenue", "satisfaction"], hue="product", corner=True, plot_kws={"alpha": 0.5, "s": 20})  # Plot pairwise relationships and color by product.
pair_grid.fig.suptitle("Pair Plot of Key Sales Variables", y=1.02)  # Add a figure-level title above the grid.
plt.show()  # Display the full pair plot.

## 20. KDE Plot: Smooth Density

A Kernel Density Estimate (KDE) approximates the distribution of numeric data with a smooth curve. It is useful to compare distribution shapes without relying only on bin boundaries.

KDE depends on smoothing bandwidth and can imply density in ranges where little or no data exists. For small samples or discrete variables, combine it with raw points or a histogram.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))  # Create a figure and axis.
sns.kdeplot(data=df, x="revenue", hue="product", common_norm=False, fill=False, ax=ax)  # Estimate a separate revenue density for each product.
ax.set_title("Revenue Density by Product")  # Add the title.
ax.set_xlabel("Revenue (sample currency)")  # Label the variable.
ax.set_ylabel("Estimated density")  # Explain the y-axis.
fig.tight_layout()  # Adjust chart spacing.
plt.show()  # Display the density curves.

## 21. ECDF Plot: Cumulative Distribution

An empirical cumulative distribution function (ECDF) shows the proportion of observations less than or equal to each value. It makes questions such as “What share of orders are below 1,000?” easy to answer visually.

Unlike a histogram, an ECDF does not require bin choices. The y-axis is a cumulative proportion from zero toward one.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))  # Create the figure and axis.
sns.ecdfplot(data=df, x="revenue", ax=ax)  # Plot the cumulative fraction of transactions by revenue.
ax.axvline(1000, linestyle="--", linewidth=1.5, label="Revenue = 1000")  # Mark a business threshold with a vertical line.
ax.set_title("Cumulative Distribution of Revenue")  # Add a descriptive title.
ax.set_xlabel("Revenue (sample currency)")  # Label the x-axis.
ax.set_ylabel("Cumulative proportion")  # Label the fraction of observations.
ax.legend()  # Display the threshold label.
fig.tight_layout()  # Fit the labels.
plt.show()  # Display the ECDF.

## 22. Area Chart: Cumulative or Component Trend

An area chart emphasizes magnitude over an ordered x-axis. It is often used for time-series totals or for showing how components contribute to a combined total.

Stacked area charts can become hard to compare when several components are present. Use clear labels and be cautious when values can be negative.

In [ ]:
daily_channel = df.pivot_table(index="date", columns="channel", values="revenue", aggfunc="sum", fill_value=0).sort_index()  # Calculate daily revenue by channel.
daily_channel = daily_channel.rolling(window=7, min_periods=1).mean()  # Smooth daily channel revenue with a seven-row rolling mean.
fig, ax = plt.subplots(figsize=(10, 4))  # Create a wide time-series figure.
daily_channel.plot.area(ax=ax, alpha=0.75)  # Draw stacked areas for the channel contributions.
ax.set_title("Smoothed Daily Revenue by Channel")  # Add a descriptive title.
ax.set_xlabel("Date")  # Label the time axis.
ax.set_ylabel("Revenue (sample currency)")  # Label the metric.
ax.legend(title="Channel")  # Identify each channel area.
fig.tight_layout()  # Adjust figure spacing.
plt.show()  # Display the area chart.

## 23. Error Bars

Error bars show uncertainty or variability around an estimate. Depending on the analysis, they may represent standard deviation, standard error, confidence intervals, or a known measurement tolerance.

Always state what the bars mean. Different interval types answer different questions and should not be treated as interchangeable.

In [ ]:
summary = df.groupby("product")["satisfaction"].agg(["mean", "std", "count"])  # Calculate mean, standard deviation, and sample size by product.
summary["se"] = summary["std"] / np.sqrt(summary["count"])  # Calculate the standard error of each product mean.
fig, ax = plt.subplots(figsize=(8, 4))  # Create the figure and axis.
ax.errorbar(summary.index, summary["mean"], yerr=summary["se"], fmt="o", capsize=5)  # Plot means with plus/minus one standard error.
ax.set_title("Mean Satisfaction by Product (±1 SE)")  # Clearly state the meaning of the error bars.
ax.set_xlabel("Product")  # Label the categories.
ax.set_ylabel("Mean satisfaction score")  # Label the mean metric.
ax.set_ylim(1, 5)  # Restrict the axis to the scale's valid range.
ax.grid(axis="y", alpha=0.25)  # Add light horizontal guides.
fig.tight_layout()  # Fit labels.
plt.show()  # Display the error-bar chart.

## 24. Annotations and Reference Lines

Annotations draw attention to an important value, threshold, or event. A horizontal or vertical reference line can show a target, average, or policy threshold.

Use annotations selectively. Too many labels distract readers; annotate the most relevant point and explain why it matters.

In [ ]:
product_summary = df.groupby("product", as_index=False)["revenue"].sum().sort_values("revenue", ascending=False)  # Aggregate product revenue and rank it.
fig, ax = plt.subplots(figsize=(8, 4))  # Create the chart and axis.
ax.bar(product_summary["product"], product_summary["revenue"])  # Draw product revenue bars.
average_revenue = product_summary["revenue"].mean()  # Calculate the average total revenue across products.
ax.axhline(average_revenue, linestyle="--", linewidth=1.5, label="Average product revenue")  # Add the average as a reference line.
top_row = product_summary.iloc[0]  # Retrieve the highest-revenue product.
ax.annotate(f"Top: {top_row['product']}", xy=(top_row["product"], top_row["revenue"]), xytext=(0, 12), textcoords="offset points", ha="center")  # Label the highest bar.
ax.set_title("Product Revenue with Average Reference")  # Add a descriptive title.
ax.set_ylabel("Revenue (sample currency)")  # Label the metric.
ax.legend()  # Show the reference-line label.
fig.tight_layout()  # Make room for the annotation and legend.
plt.show()  # Display the annotated chart.

## 25. Multiple Charts with Subplots

Subplots arrange multiple charts in one figure so readers can compare related views. Each subplot has its own `Axes` object, which controls titles, labels, ticks, and drawing.

Keep the scales comparable when direct visual comparison matters. Use a shared axis only when the variables have compatible units and ranges.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))  # Create one figure with two axes side by side.
axes[0].hist(df["revenue"], bins=18, edgecolor="white")  # Draw a revenue distribution in the first axis.
axes[0].set_title("Revenue Distribution")  # Title the first panel.
axes[0].set_xlabel("Revenue")  # Label the first panel's x-axis.
axes[0].set_ylabel("Count")  # Label the first panel's y-axis.
product_totals = df.groupby("product")["revenue"].sum().sort_values(ascending=False)  # Calculate revenue totals by product.
axes[1].bar(product_totals.index, product_totals.values)  # Draw product revenue in the second axis.
axes[1].set_title("Revenue by Product")  # Title the second panel.
axes[1].set_xlabel("Product")  # Label the second panel's x-axis.
axes[1].set_ylabel("Revenue")  # Label the second panel's y-axis.
axes[1].tick_params(axis="x", rotation=30)  # Rotate labels if needed to prevent overlap.
fig.suptitle("Sales Overview")  # Add a title for the full figure.
fig.tight_layout()  # Arrange panel spacing automatically.
plt.show()  # Display both charts together.

## 26. Formatting Titles, Labels, and Ticks

Presentation quality matters. A chart should answer a question, use descriptive labels and units, format tick marks clearly, and avoid overlapping text.

Matplotlib lets you control font size, label rotation, grid lines, axis limits, and number formats. Formatting should improve interpretation rather than distract from the data.

In [ ]:
monthly = df.set_index("date")["revenue"].resample("MS").sum()  # Aggregate transaction revenue into monthly totals.
fig, ax = plt.subplots(figsize=(9, 4))  # Create a chart with room for date labels.
ax.plot(monthly.index, monthly.values, marker="o", linewidth=2)  # Plot the monthly totals with visible point markers.
ax.set_title("Monthly Revenue", fontsize=15, pad=12)  # Set a readable title and padding.
ax.set_xlabel("Month", fontsize=11)  # Set the x-axis label.
ax.set_ylabel("Revenue (sample currency)", fontsize=11)  # State the metric and its units.
ax.tick_params(axis="x", labelrotation=30, labelsize=9)  # Rotate and resize date tick labels.
ax.ticklabel_format(axis="y", style="plain")  # Avoid scientific notation on the revenue axis.
ax.grid(axis="y", linestyle=":", alpha=0.35)  # Add a subtle dotted horizontal grid.
fig.tight_layout()  # Prevent labels from clipping.
plt.show()  # Display the formatted chart.

## 27. Applying a Chart Style

Matplotlib styles and Seaborn themes make multiple charts look consistent. A shared visual system is useful for reports and dashboards.

A style should not replace good labeling or appropriate chart choice. Avoid overly decorative effects, especially when they make comparisons harder.

In [ ]:
sns.set_theme(style="whitegrid", context="notebook")  # Apply a consistent Seaborn theme to subsequent charts.
fig, ax = plt.subplots(figsize=(8, 4))  # Create the figure and axis.
sns.barplot(data=df, x="region", y="profit", estimator="sum", errorbar=None, color=plt.rcParams["axes.prop_cycle"].by_key()["color"][0], ax=ax)  # Compare total profit by region without uncertainty intervals.
ax.set_title("Total Profit by Region")  # Add a descriptive title.
ax.set_xlabel("Region")  # Label the category axis.
ax.set_ylabel("Profit (sample currency)")  # Label the numeric measure.
fig.tight_layout()  # Adjust label spacing.
plt.show()  # Display the chart using the chosen theme.

## 28. Log Scale for Skewed Values

A logarithmic axis can make values spanning several orders of magnitude easier to compare. It compresses large values more than small values.

Log axes cannot display zero or negative values. Use a log scale only when it matches the data and audience; explain it clearly because equal vertical distances represent multiplicative, not additive, changes.

In [ ]:
positive_revenue = df.loc[df["revenue"] > 0, "revenue"]  # Keep only positive values because a log axis cannot display zero or negative values.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))  # Create two panels to compare scales.
axes[0].hist(positive_revenue, bins=18, edgecolor="white")  # Plot the usual revenue distribution.
axes[0].set_title("Linear Scale")  # Label the first panel's scale.
axes[0].set_xlabel("Revenue")  # Label the first x-axis.
axes[0].set_ylabel("Count")  # Label the first y-axis.
axes[1].hist(positive_revenue, bins=18, edgecolor="white")  # Plot the same revenue values in the second panel.
axes[1].set_yscale("log")  # Use a logarithmic count axis to emphasize differences among small counts.
axes[1].set_title("Logarithmic Count Scale")  # Explain the changed axis scale.
axes[1].set_xlabel("Revenue")  # Label the second x-axis.
axes[1].set_ylabel("Count (log scale)")  # Explicitly identify the logarithmic y-axis.
fig.tight_layout()  # Space the panels.
plt.show()  # Display both histograms.

## 29. Saving Figures

Save charts for reports, presentations, or web pages with `savefig()`. Common formats include PNG, SVG, and PDF.

Use `bbox_inches="tight"` to reduce clipping of labels. Higher DPI improves raster-image sharpness; SVG and PDF are vector formats that scale well. This example saves into the current notebook working directory.

In [ ]:
daily_profit = df.groupby("date")["profit"].sum().sort_index()  # Aggregate profit by date in chronological order.
fig, ax = plt.subplots(figsize=(9, 4))  # Create a figure for saving.
ax.plot(daily_profit.index, daily_profit.values, linewidth=1.8)  # Plot daily profit totals.
ax.set_title("Daily Profit")  # Add a descriptive title.
ax.set_xlabel("Date")  # Label the time axis.
ax.set_ylabel("Profit (sample currency)")  # Label the metric.
fig.tight_layout()  # Adjust spacing before export.
fig.savefig("daily_profit.png", dpi=160, bbox_inches="tight")  # Save a clear PNG image in the current working directory.
fig.savefig("daily_profit.svg", bbox_inches="tight")  # Save a scalable vector copy.
plt.show()  # Display the chart in the notebook.
print("Saved daily_profit.png and daily_profit.svg in the notebook's working directory.")  # Confirm the expected output filenames.

## 30. Mini Project: Build a Sales Dashboard

This mini project combines aggregation and visualization into a compact four-panel dashboard:

1. Daily revenue trend.
2. Revenue by product.
3. Revenue by region and channel.
4. Satisfaction distribution.

A dashboard should prioritize a small set of useful business questions. Keep the titles, units, and visual encodings consistent, and avoid interpreting synthetic example data as real business evidence.

In [ ]:
daily_totals = df.groupby("date", as_index=False)["revenue"].sum().sort_values("date")  # Prepare daily revenue for the trend panel.
product_totals = df.groupby("product")["revenue"].sum().sort_values(ascending=False)  # Prepare ranked revenue by product.
region_channel_totals = df.pivot_table(index="region", columns="channel", values="revenue", aggfunc="sum", fill_value=0)  # Prepare regional revenue split by channel.
fig, axes = plt.subplots(2, 2, figsize=(13, 9))  # Create a dashboard with two rows and two columns.
axes[0, 0].plot(daily_totals["date"], daily_totals["revenue"], linewidth=1.5)  # Draw the daily revenue trend.
axes[0, 0].set_title("Daily Revenue Trend")  # Title the trend panel.
axes[0, 0].set_xlabel("Date")  # Label the trend's x-axis.
axes[0, 0].set_ylabel("Revenue")  # Label the trend's y-axis.
axes[0, 0].tick_params(axis="x", rotation=35)  # Rotate date labels for readability.
axes[0, 1].bar(product_totals.index, product_totals.values)  # Draw the product revenue ranking.
axes[0, 1].set_title("Revenue by Product")  # Title the product panel.
axes[0, 1].set_xlabel("Product")  # Label the product categories.
axes[0, 1].set_ylabel("Revenue")  # Label the product metric.
region_channel_totals.plot(kind="bar", stacked=True, ax=axes[1, 0])  # Draw stacked regional revenue by channel.
axes[1, 0].set_title("Revenue by Region and Channel")  # Title the regional composition panel.
axes[1, 0].set_xlabel("Region")  # Label the regional categories.
axes[1, 0].set_ylabel("Revenue")  # Label the regional metric.
axes[1, 0].tick_params(axis="x", rotation=0)  # Keep region labels horizontal.
sns.histplot(data=df, x="satisfaction", bins=10, color=plt.rcParams["axes.prop_cycle"].by_key()["color"][0], ax=axes[1, 1])  # Draw the customer satisfaction distribution.
axes[1, 1].set_title("Satisfaction Scores")  # Title the satisfaction panel.
axes[1, 1].set_xlabel("Satisfaction score")  # Label the score axis.
axes[1, 1].set_ylabel("Transactions")  # Label the count axis.
fig.suptitle("Sales Analysis Dashboard", fontsize=16)  # Add a title spanning all four panels.
fig.tight_layout()  # Adjust spacing so the panels do not overlap.
plt.show()  # Display the completed dashboard.